# 01. 데이터 다운로드

HF `datasets`의 **`ksang/steamreviews`** (Kaggle Steam Reviews 미러, `review_text` + `review_score` 1/-1)에서
리뷰를 받아 원본 샘플을 `data/raw_sample.csv`로 저장하고 기초 EDA를 수행한다.

- 앞부분만 자르면 특정 게임(app_id 순 정렬)에 편중되므로 **셔플 후 샘플링**한다.
- 정제·분할은 `02_preprocessing.ipynb`에서 진행한다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 import 경로에 추가 (notebooks/ 하위에서 실행되므로)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from datasets import load_dataset
from src.config import DATA_DIR, RANDOM_SEED

DATASET_ID = "ksang/steamreviews"
LIMIT = 20000

In [2]:
ds = load_dataset(DATASET_ID, split="train")
print(f"전체 리뷰 수: {len(ds):,}")
sample = ds.shuffle(seed=RANDOM_SEED).select(range(LIMIT))
df = sample.to_pandas()
df.head(3)

Repo card metadata block was not found. Setting CardData to empty.


전체 리뷰 수: 6,417,106


,app_id,app_name,review_text,review_score,review_votes
0,239200,Amnesia: A Machine for Pigs,"Little game, such shame, much disappoint, not ...",-1,0
1,27940,Dead Horde,10/10 GOTY 2014 Pros: -Shoot dead people. -Blo...,-1,0
2,22100,Mount & Blade,Now I'm not saying I know how to play this gam...,1,1


In [ ]:
# 원본 샘플 저장 (전처리는 02 노트북에서)
DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_DIR / "raw_sample.csv", index=False)
print(f"raw_sample.csv 저장: {len(df):,}건")
print(df["review_score"].value_counts(normalize=True).rename({1: "추천", -1: "비추천"}))

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

counts = df["review_score"].value_counts().rename({1: "추천(긍정)", -1: "비추천(부정)"})
ax1.bar(counts.index, counts.values, color=["#4c72b0", "#c44e52"])
ax1.set_title("원본 라벨 분포")
ax1.set_ylabel("리뷰 수")
for i, v in enumerate(counts.values):
    ax1.text(i, v, f"{v:,}", ha="center", va="bottom")

lengths = df["review_text"].astype(str).str.split().str.len()
ax2.hist(lengths.clip(upper=300), bins=50, color="#55a868")
ax2.set_title("원본 리뷰 길이 분포 (단어 수, 300+ 절단)")
ax2.set_xlabel("단어 수")
ax2.set_ylabel("리뷰 수")

plt.tight_layout()
plt.show()